In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import torch
import psutil
import sys
import os
from matplotlib.ticker import ScalarFormatter
import yaml

In [2]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.default'] = 'rm'

plt.rc("font", family="serif", size=30)
plt.rc("axes", titlesize="medium")

plt.rcParams['xtick.labelsize'] = 30
plt.rcParams['ytick.labelsize'] = 30

plt.rcParams["axes.formatter.limits"] = [-3,3]

rect_double    = (0.05, 0.12, 0.98, 0.97) # left, bottom, right, top
rect_double_with_legend = (0.14, 0.12, 0.98, 0.97) # left, bottom, right, top

In [3]:
cd C:\Users\flori\OneDrive\Daten\Promotion\Machine Learning\CaloINN\src

C:\Users\flori\OneDrive\Daten\Promotion\Machine Learning\CaloINN\src


c:\Users\flori\anaconda3\envs\CaloINN\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
from documenter import Documenter
from trainer import VAETrainer
import data_util
import plotting

In [5]:
print(psutil.virtual_memory())
torch.set_default_dtype(torch.float32)

svmem(total=17005826048, available=9272659968, percent=45.5, used=7733166080, free=9272659968)


In [6]:
def load_trainer(directory, use_cuda=True):
    use_cuda = torch.cuda.is_available() and use_cuda
    device = 'cuda:0' if use_cuda else 'cpu' 


    with open(os.path.join(directory, r"params.yaml")) as f:
        params = yaml.load(f, Loader=yaml.FullLoader)
        
    doc = Documenter(params['run_name'], existing_run=True, basedir=directory,
                    log_name="log_jupyter.txt", read_only=True)
        
    trainer = VAETrainer(params, device, doc)
    
    return trainer, params, device, doc

In [7]:
trainer, params, device, doc = load_trainer(r"..\results\test")

Using the directory: ..\results\test
Device:  cpu
Using layers tensor([ 0,  1,  2,  3, 12], dtype=torch.int32)
Fixed 1 negative layers
Removed 11 of 127271 events (0.01%)
Number of parameters 36234
CVAE(
  (encoder): Sequential(
    (fc0): Linear(in_features=6247, out_features=1, bias=True)
    (relu0): ReLU()
    (fc1): Linear(in_features=1, out_features=1, bias=True)
    (relu1): ReLU()
    (fc_mu_logvar): Linear(in_features=1, out_features=2000, bias=True)
  )
  (decoder): Sequential(
    (fc0): Linear(in_features=1007, out_features=1, bias=True)
    (relu0): ReLU()
    (fc1): Linear(in_features=1, out_features=1, bias=True)
    (relu1): ReLU()
    (fc_out): Linear(in_features=1, out_features=6240, bias=True)
  )
  (norm_x_in): LearnableNorm()
  (norm_x_out): LearnableNorm()
)


In [8]:
x, c = trainer.test_loader.data, trainer.test_loader.cond



model = trainer.model

with torch.no_grad():
    x = model._preprocess_encoding(x, c)[..., :-7]
    x = model.norm_x_out( (x, ), rev=True)[0][0]
    x = model.logit_trafo_out(x)
    x[x<0] = 0
    x = data_util.unnormalize_layers(x, c, model.layer_boundaries, eps=model.eps, noise_width=None)
    if model.threshold is not None:
        x[x < model.threshold] = 0
    else:
        x[x < 0] = 0



    # Plotting
    data = trainer.test_loader.data
    cond = trainer.test_loader.cond
    generated = x
    data_post, cond_post, layer_boundaries_post = data_util.postprocess(data, cond, trainer.layer_boundaries, trainer.negative_layers)
    generated_post, _, _ = data_util.postprocess(generated, cond, trainer.layer_boundaries, trainer.negative_layers)
    params = plotting.get_plot_params(layer_boundaries_post, trainer.coordinates.cpu().numpy(), used_layers=trainer.params.get("used_calo_layers", None))
    subdir = os.path.join("plots", f'test')
    plot_dir = trainer.doc.get_file(subdir)


    plotting.plot_all_hist([data_post.cpu().numpy(), generated_post.cpu().numpy()], 
                            [cond_post.cpu().numpy(), cond_post.cpu().numpy()], 
                            params, plot_dir=plot_dir, summary_plot=True,
                            summary_plot_name="summary.pdf", errorbars_true=True,
                            errorbars_fake=True, ncol=5)
